Load Data

In [87]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Settings for Pandas and visualization

In [88]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
sns.set_style('whitegrid')
sns.set_palette('viridis')

Loading CC_Calls cleaned data

In [89]:
df_cc_calls = pd.read_csv("../../../data/interim/cc_calls_cleaned.csv", )
df_billings = pd.read_csv('../../../data/interim/billings_features.csv')

df_renewal_dates = df_billings[['co_ref', 'renewal_month']].copy()

print(f"CC calls shape : {df_cc_calls.shape}")
print(f"Unique customers    : {df_cc_calls['co_ref'].nunique()}")

CC calls shape : (31636, 30)
Unique customers    : 14988


Parse Dates

In [90]:
df_cc_calls['call_date'] = pd.to_datetime(df_cc_calls['call_date'], errors='coerce')

df_renewal_dates['prospect_renewal_date'] = pd.to_datetime(
    df_renewal_dates['renewal_month'], errors='coerce'
)

Creating Feature for Cutoff Date

In [91]:
df_renewal_dates['cutoff_date'] = df_renewal_dates['prospect_renewal_date'] - pd.Timedelta(days=14)

Rename call_year → renewal_year => Ensures cc_calls can be joined to billings on co_ref + renewal_year.

In [92]:
# Extract year
df_cc_calls['renewal_year'] = df_cc_calls['call_date'].dt.year
print(df_cc_calls['renewal_year'].value_counts().sort_index())

renewal_year
2024     5365
2025    25588
2026      683
Name: count, dtype: int64


Merge CC_calls Date onto calls

In [93]:
df_cc_calls = df_cc_calls.merge(
    df_renewal_dates,
    on='co_ref',
    how='inner'
)

print(f"Calls with cc_calls date matched: {df_cc_calls['prospect_renewal_date'].notna().sum()}")
print(f"Calls with no match            : {df_cc_calls['prospect_renewal_date'].isna().sum()}")

Calls with cc_calls date matched: 88830
Calls with no match            : 0


Apply 14 Days filter

In [94]:
df_cc_calls = df_cc_calls[
    df_cc_calls['call_date'] <= df_cc_calls['cutoff_date']
]

df_cc_calls_14 = df_cc_calls.copy()

print(f"Total calls              : {len(df_cc_calls):,}")
print(f"Calls in 14-day window   : {len(df_cc_calls_14):,}")
print(f"Calls outside window     : {len(df_cc_calls) - len(df_cc_calls_14):,}")
print(f"Customers with 14d calls : {df_cc_calls_14['co_ref'].nunique():,}")

Total calls              : 9,086
Calls in 14-day window   : 9,086
Calls outside window     : 0
Customers with 14d calls : 5,003


Convert Yes/No/Unknown Columns to Binary

In [95]:
# Yes = 1, No = 0, Unknown = 0 (no confirmed signal)

yes_no_cols = [
    'cc_care_package_discussed',
    'cc_urgency_getting_on_site',
    'cc_external_consultant',
    'cc_agent_cross_sell_attempt',
    'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship',
    'cc_questionnaire_completion',
    'cc_chasing_response',
    'cc_issues_within_questionnaire',
    'cc_login_issues',
    'cc_platform_issues',
    'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support',
    'cc_pricing_mentioned',
    'cc_pricing_sentiment_impact',
    'cc_refund_discussed',
    'cc_contractor_suggest_leave',
    'cc_contractor_complained',
]

for col in yes_no_cols:
    df_cc_calls_14[col] = df_cc_calls_14[col].map({'Yes': 1, 'No': 0, 'Unknown': 0})

Encode Sentiment Column

In [96]:
# Dissatisfied = -1, Neutral = 0, Satisfied = 1
sentiment_map = {
    'Satisfied'    :  1,
    'Neutral'      :  0,
    'Not Discussed':  0,
    'Dissatisfied' : -1,
    'Unknown'      :  0,
}
 
df_cc_calls_14['cc_sentiment_encoded'] = df_cc_calls_14['cc_contractor_sentiment'].map(sentiment_map)
df_cc_calls_14['cc_dissatisfied_flag'] = (df_cc_calls_14['cc_contractor_sentiment'] == 'Dissatisfied').astype(int)

Encoding Direction Column

In [97]:
# IN_BOUND = 1, OUT_BOUND = 0
df_cc_calls_14['is_inbound'] = (df_cc_calls_14['direction'] == 'IN_BOUND').astype(int)

One hot encoding

In [98]:
ohe_cols = ['cc_care_package', 'cc_call_initiated_by']

for col in ohe_cols:
    df_cc_calls_14[col] = (
        df_cc_calls_14[col]
        .str.lower()
        .str.replace(' ', '_')
    )
    
df_cc_calls_14 = pd.get_dummies(
    df_cc_calls_14,
    columns=ohe_cols,
    drop_first=False,
    dtype=int
)

Feature engineering


In [99]:
# Sentiment improved or worsened during the call
df_cc_calls_14['sentiment_improved'] = (
    df_cc_calls_14['cc_contractor_sentiment_end_score'] > 
    df_cc_calls_14['cc_contractor_sentiment_start_score']
).astype(int)
 
df_cc_calls_14['sentiment_worsened'] = (
    df_cc_calls_14['cc_contractor_sentiment_end_score'] < 
    df_cc_calls_14['cc_contractor_sentiment_start_score']
).astype(int)
 
# Sentiment change score
df_cc_calls_14['sentiment_change'] = (
    df_cc_calls_14['cc_contractor_sentiment_end_score'] - 
    df_cc_calls_14['cc_contractor_sentiment_start_score']
)
 
# High risk call — multiple issues in one call
df_cc_calls_14['cc_high_risk_call'] = (
    (df_cc_calls_14['cc_contractor_complained']          == 1) |
    (df_cc_calls_14['cc_contractor_suggest_leave']        == 1) |
    (df_cc_calls_14['cc_business_struggles_financial_hardship'] == 1) |
    (df_cc_calls_14['cc_refund_discussed']               == 1)
).astype(int)
 
# Dissatisfaction issue count per call
dissatisfaction_cols = [
    'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support',
    'cc_platform_issues',
    'cc_login_issues',
]
df_cc_calls_14['cc_dissatisfaction_issue_count'] = df_cc_calls_14[dissatisfaction_cols].sum(axis=1)
 
# Pricing pressure flag — pricing mentioned and had sentiment impact
df_cc_calls_14['pricing_pressure_flag'] = (
    (df_cc_calls_14['cc_pricing_mentioned']        == 1) &
    (df_cc_calls_14['cc_pricing_sentiment_impact'] == 1)
).astype(int)

Aggregate Per Customer using co_ref

In [100]:
agg_dict = {
    # Call counts
    'contact_id'                                    : 'count',
    'is_inbound'                                    : 'sum',

    # Sentiment
    'cc_sentiment_encoded'                          : 'mean',
    'cc_dissatisfied_flag'                          : 'max',
    'cc_contractor_sentiment_start_score'           : 'mean',
    'cc_contractor_sentiment_end_score'             : 'mean',
    'cc_contractor_sentiment_overall_score'         : 'mean',

    # Issue flags
    'cc_customer_issues_concerns'                   : 'max',
    'cc_business_struggles_financial_hardship'      : 'max',
    'cc_login_issues'                               : 'max',
    'cc_platform_issues'                            : 'max',
    'cc_dissatisfaction_time_to_complete'           : 'max',
    'cc_process_complexity_concerns'                : 'max',
    'cc_questions_harder_than_expected'             : 'max',
    'cc_dissatisfaction_support'                    : 'max',
    'cc_pricing_mentioned'                          : 'max',
    'cc_pricing_sentiment_impact'                   : 'max',
    'cc_refund_discussed'                           : 'max',
    'cc_contractor_suggest_leave'                   : 'max',
    'cc_contractor_complained'                      : 'max',
    'cc_urgency_getting_on_site'                    : 'max',
    'cc_chasing_response'                           : 'max',
    'cc_questionnaire_completion'                   : 'max',
    'cc_care_package_discussed'                     : 'max',
    'cc_external_consultant'                        : 'max',
    'cc_agent_cross_sell_attempt'                   : 'max',
    'cc_issues_within_questionnaire'                : 'max',

    # Engineered features
    'sentiment_improved'                            : 'max',
    'sentiment_worsened'                            : 'max',
    'sentiment_change'                              : 'mean',
    'cc_high_risk_call'                                : 'max',
    'cc_dissatisfaction_issue_count'                   : 'sum',
    'pricing_pressure_flag'                         : 'max',

    # OHE — care package type
    'cc_care_package_assisted'                      : 'max',
    'cc_care_package_express'                       : 'max',
    'cc_care_package_not_discussed'                 : 'max',
    'cc_care_package_premier'                       : 'max',
    'cc_care_package_standard'                      : 'max',
    'cc_care_package_unknown'                       : 'max',

    # OHE — call initiated by
    'cc_call_initiated_by_agent'                    : 'sum',
    'cc_call_initiated_by_customer'                 : 'sum',
    'cc_call_initiated_by_not_relevant'             : 'sum',
    'cc_call_initiated_by_unknown'                  : 'sum',
}

df_cc_calls_agg = df_cc_calls_14.groupby(['co_ref', 'renewal_year']).agg(agg_dict).reset_index()

df_cc_calls_agg.rename(columns={
    'contact_id' : 'cc_calls_14d_total_calls',
    'is_inbound' : 'cc_calls_14d_inbound_calls',
}, inplace=True)

 
print(f"Shape after aggregation: {df_cc_calls_agg.shape}")
df_cc_calls_agg.head()

Shape after aggregation: (5321, 45)


,co_ref,renewal_year,cc_calls_14d_total_calls,cc_calls_14d_inbound_calls,cc_sentiment_encoded,cc_dissatisfied_flag,cc_contractor_sentiment_start_score,cc_contractor_sentiment_end_score,cc_contractor_sentiment_overall_score,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,cc_login_issues,cc_platform_issues,cc_dissatisfaction_time_to_complete,cc_process_complexity_concerns,cc_questions_harder_than_expected,cc_dissatisfaction_support,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,cc_urgency_getting_on_site,cc_chasing_response,cc_questionnaire_completion,cc_care_package_discussed,cc_external_consultant,cc_agent_cross_sell_attempt,cc_issues_within_questionnaire,sentiment_improved,sentiment_worsened,sentiment_change,cc_high_risk_call,cc_dissatisfaction_issue_count,pricing_pressure_flag,cc_care_package_assisted,cc_care_package_express,cc_care_package_not_discussed,cc_care_package_premier,cc_care_package_standard,cc_care_package_unknown,cc_call_initiated_by_agent,cc_call_initiated_by_customer,cc_call_initiated_by_not_relevant,cc_call_initiated_by_unknown
0,AA0784,2025,1,0,1.0,0,50.0,80.0,80.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,30.0,0,0,0,0,0,1,0,0,0,0,1,0,0
1,AA0923,2025,1,1,1.0,0,50.0,90.0,90.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,40.0,0,0,0,0,0,1,0,0,0,0,1,0,0
2,AA0994,2024,2,2,1.0,0,65.0,90.0,87.5,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,25.0,0,1,0,0,0,1,0,0,0,0,2,0,0
3,AA1155,2025,1,0,1.0,0,80.0,90.0,85.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,10.0,0,0,0,0,0,0,0,1,0,1,0,0,0
4,AA1392,2024,1,0,0.0,0,50.0,80.0,80.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,30.0,0,0,0,0,0,1,0,0,0,1,0,0,0


Add Outbound Calls Column

In [101]:
df_cc_calls_agg['cc_calls_14d_outbound_calls'] = (
    df_cc_calls_agg['cc_calls_14d_total_calls'] - df_cc_calls_agg['cc_calls_14d_inbound_calls']
)

Final check for nulls and shape

In [102]:
print("Nulls:")
print(df_cc_calls_agg.isnull().sum()[df_cc_calls_agg.isnull().sum() > 0])
 
print(f"\nShape: {df_cc_calls_agg.shape}")
print(f"Unique customers: {df_cc_calls_agg['co_ref'].nunique()}")
print(f"Unique co_ref + renewal_year : {df_cc_calls_agg.drop_duplicates(subset=['co_ref','renewal_year']).shape[0]}")

df_cc_calls_agg.describe()

Nulls:
Series([], dtype: int64)

Shape: (5321, 46)
Unique customers: 5003
Unique co_ref + renewal_year : 5321


,renewal_year,cc_calls_14d_total_calls,cc_calls_14d_inbound_calls,cc_sentiment_encoded,cc_dissatisfied_flag,cc_contractor_sentiment_start_score,cc_contractor_sentiment_end_score,cc_contractor_sentiment_overall_score,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,cc_login_issues,cc_platform_issues,cc_dissatisfaction_time_to_complete,cc_process_complexity_concerns,cc_questions_harder_than_expected,cc_dissatisfaction_support,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,cc_urgency_getting_on_site,cc_chasing_response,cc_questionnaire_completion,cc_care_package_discussed,cc_external_consultant,cc_agent_cross_sell_attempt,cc_issues_within_questionnaire,sentiment_improved,sentiment_worsened,sentiment_change,cc_high_risk_call,cc_dissatisfaction_issue_count,pricing_pressure_flag,cc_care_package_assisted,cc_care_package_express,cc_care_package_not_discussed,cc_care_package_premier,cc_care_package_standard,cc_care_package_unknown,cc_call_initiated_by_agent,cc_call_initiated_by_customer,cc_call_initiated_by_not_relevant,cc_call_initiated_by_unknown,cc_calls_14d_outbound_calls
count,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.00000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.00000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000,5321.000000
mean,2024.321368,1.707574,0.392407,0.482985,0.043977,52.68411,79.077774,81.606856,0.146025,0.042473,0.074046,0.109378,0.066341,0.132306,0.006766,0.030445,0.175531,0.046420,0.008645,0.030070,0.097726,0.135689,0.285473,0.267995,0.293930,0.137944,0.063146,0.178350,0.888367,0.013343,26.393665,0.137192,0.508175,0.045856,0.009397,0.102612,0.824469,0.024807,0.15655,0.010148,0.442774,1.153731,0.100733,0.010336,1.315166
std,0.467046,1.193890,0.775646,0.510953,0.205063,11.25233,14.811037,10.668249,0.353165,0.201685,0.261871,0.312142,0.248900,0.338855,0.081983,0.171826,0.380456,0.210412,0.092584,0.170795,0.296972,0.342490,0.451682,0.442956,0.455603,0.344874,0.243248,0.382843,0.314944,0.114751,15.120183,0.344083,1.137195,0.209192,0.096489,0.303480,0.380456,0.155552,0.36341,0.100237,0.698121,1.155589,0.398778,0.130378,1.218472
min,2024.000000,1.000000,0.000000,-1.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-50.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2024.000000,1.000000,0.000000,0.000000,0.000000,50.00000,76.666667,80.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,15.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,2024.000000,1.000000,0.000000,0.500000,0.000000,50.00000,80.000000,81.666667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,30.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.00000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000
75%,2025.000000,2.000000,1.000000,1.000000,0.000000,50.00000,90.000000,90.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0

Saving final features dataset

In [103]:
df_cc_calls_agg.to_csv('../../../data/interim/cc_calls_features.csv', index=False)
print("Saved => data/interim/cc_calls_features.csv")

Saved => data/interim/cc_calls_features.csv


### Hypothesis

Hypothesis 1 => Customers with higher number of issues are more likely to churn

Hypothesis 2 => Customers with lower sentiment scores tend to churn

Hypothesis 3 => Customers who complain more frequently are more likely to churn

Hypothesis 4 => Customers with fewer interactions (calls) are more likely to churn

Hypothesis 5 => Negative experience over time (low sentiment + high issues) increases churn probability